# ClassificationProgressiveLearner & LifelongClassificationForest

## TreeClassificationTransformer using treeple

In [22]:
from tensorflow import keras
import numpy as np
#from sklearn.tree import DecisionTreeClassifier
from treeple.tree import DecisionTreeClassifier
from sklearn.utils.validation import check_array, check_is_fitted, check_X_y
from sktree import ObliqueRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from proglearn.base import BaseTransformer

class TreeClassificationTransformer(BaseTransformer):
    """
    A class used to transform data from a category to a specialized representation.

    Parameters
    ----------
    kwargs : dict, default={}
        A dictionary to contain parameters of the tree.

    Attributes
    ----------
    transformer : sklearn.tree.DecisionTreeClassifier
        an internal sklearn DecisionTreeClassifier
    """

    def __init__(self, kwargs={}):
        self.kwargs = kwargs

    def fit(self, X, y):
        """
        Fits the transformer to data X with labels y.

        Parameters
        ----------
        X : ndarray
            Input data matrix.
        y : ndarray
            Output (i.e. response data matrix).

        Returns
        -------
        self : TreeClassificationTransformer
            The object itself.
        """
        X, y = check_X_y(X, y)
        self.transformer_ = DecisionTreeClassifier(**self.kwargs).fit(X, y)
        return self

    def transform(self, X):
        """
        Performs inference using the transformer.

        Parameters
        ----------
        X : ndarray
            Input data matrix.

        Returns
        -------
        X_transformed : ndarray
            The transformed input.

        Raises
        ------
        NotFittedError
            When the model is not fitted.
        """
        X = check_array(X)
        check_is_fitted(self)
        return self.transformer_.apply(X)

## LifelongClassificationForest

In [23]:
from proglearn.progressive_learner import ClassificationProgressiveLearner
#from proglearn.transformers import TreeClassificationTransformer
from proglearn.voters import TreeClassificationVoter
from proglearn.deciders import SimpleArgmaxAverage

import numpy as np

from sklearn.utils.validation import check_X_y, check_array

class LifelongClassificationForest(ClassificationProgressiveLearner):
    def __init__(
        self,
        default_n_estimators=100,
        default_tree_construction_proportion=0.67,
        default_kappa=np.inf,
        default_max_depth=30,
    ):
        super().__init__(
            default_transformer_class=TreeClassificationTransformer,
            default_transformer_kwargs={},
            default_voter_class=TreeClassificationVoter,
            default_voter_kwargs={"kappa": default_kappa},
            default_decider_class=SimpleArgmaxAverage,
            default_decider_kwargs={},
        )

        self.default_n_estimators = default_n_estimators
        self.default_tree_construction_proportion = default_tree_construction_proportion
        self.default_kappa = default_kappa
        self.default_max_depth = default_max_depth

    def add_task(
        self,
        X,
        y,
        task_id=None,
        n_estimators="default",
        tree_construction_proportion="default",
        kappa="default",
        max_depth="default",
    ):
        if n_estimators == "default":
            n_estimators = self.default_n_estimators
        if tree_construction_proportion == "default":
            tree_construction_proportion = self.default_tree_construction_proportion
        if kappa == "default":
            kappa = self.default_kappa
        if max_depth == "default":
            max_depth = self.default_max_depth

        X, y = check_X_y(X, y)
        return super().add_task(
            X,
            y,
            task_id=task_id,
            transformer_voter_decider_split=[
                tree_construction_proportion,
                1 - tree_construction_proportion,
                0,
            ],
            num_transformers=n_estimators,
            transformer_kwargs={"kwargs": {"max_depth": max_depth}},
            voter_kwargs={
                "classes": np.unique(y),
                "kappa": kappa,
            },
            decider_kwargs={"classes": np.unique(y)},
        )


    def add_transformer(
        self,
        X,
        y,
        transformer_id=None,
        n_estimators="default",
        max_depth="default",
    ):
        """
        adds a transformer with id transformer_id and max tree depth max_depth, trained on
        given input data matrix, X, and output data matrix, y, to the Lifelong Classification Forest.
        Also trains the voters and deciders from new transformer to previous tasks, and will
        train voters and deciders from this transformer to all new tasks.

        Parameters
        ----------
        X : ndarray
            The input data matrix.

        y : ndarray
            The output (response) data matrix.

        transformer_id : obj, default=None
            The id corresponding to the transformer being added.

        n_estimators : int or str, default='default'
            The number of trees used for the given task.

        max_depth : int or str, default='default'
            The maximum depth of a tree in the Lifelong Classification Forest.
            The default is used if 'default' is provided.

        Returns
        -------
        self : LifelongClassificationForest
            The object itself.
        """
        if n_estimators == "default":
            n_estimators = self.default_n_estimators
        if max_depth == "default":
            max_depth = self.default_max_depth

        X, y = check_X_y(X, y)
        return super().add_transformer(
            X,
            y,
            transformer_kwargs={"kwargs": {"max_depth": max_depth}},
            transformer_id=transformer_id,
            num_transformers=n_estimators,
        )


    def predict_proba(self, X, task_id):
        """
        estimates class posteriors under task_id for each example in input data X.

        Parameters
        ----------
        X : ndarray
            The input data matrix.

        task_id:
            The id corresponding to the task being mapped to.

        Returns
        -------
        y_proba_hat : ndarray of shape [n_samples, n_classes]
            posteriors per example
        """
        return super().predict_proba(check_array(X), task_id)


    def predict(self, X, task_id):
        """
        predicts class labels under task_id for each example in input data X.

        Parameters
        ----------
        X : ndarray
            The input data matrix.

        task_id : obj
            The id corresponding to the task being mapped to.

        Returns
        -------
        y_hat : ndarray of shape [n_samples]
            predicted class label per example
        """
        return super().predict(check_array(X), task_id)

## Load Data

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import nibabel as nb
from scipy import ndimage
import scipy.stats as ss
from joblib import Parallel, delayed
from proglearn.deciders import SimpleArgmaxAverage
from proglearn.progressive_learner import ProgressiveLearner
#Use New Transformer
from proglearn.voters import TreeClassificationVoter, KNNClassificationVoter

df = pd.read_excel(r'C:\Users\86199\PycharmProjects\pythonProject\BDD Transfer\Macaque.parcellated_thickness.xlsx')
df.head()
df_sex = pd.read_csv(r'C:\Users\86199\PycharmProjects\pythonProject\BDD Transfer\uwmadison.csv')
df_sex.head()
X1 = []
X2 = []
y_monkey = []
IDs = set(df['participant_id'])
ref_IDs = set(df_sex['participant_id'])

for subject in tqdm(IDs):
    if subject in ref_IDs:
        features = np.array(df[df['participant_id'] == subject]).reshape(-1)[4:]
        gender = list(df_sex[df_sex['participant_id'] == subject]['sex'])
        sex = int(gender[0] == 'F')

        X1.append(list(features[:182]))
        X2.append(list(features[182:]))
        y_monkey.append(sex)

X1_monkey = np.array(X1)
X2_monkey = np.array(X2)
imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
X1_monkey = imp_mean.fit_transform(X1_monkey)
X2_monkey = imp_mean.fit_transform(X2_monkey)
print(X1_monkey.shape, X2_monkey.shape)

df = pd.read_excel(r'C:\Users\86199\PycharmProjects\pythonProject\BDD Transfer\Human.parcellated_thickness.xlsx')
df.head()
df_sex = pd.read_excel(r'C:\Users\86199\PycharmProjects\pythonProject\BDD Transfer\subjects_age_sex_data_MRI.xlsx')
df_sex.head()
X1 = []
X2 = []
y_human = []
IDs = set(df['sid'])
ref_IDs = set(df_sex['ID'])

for subject in tqdm(IDs):
    if subject in ref_IDs:
        features = np.array(df[df['sid'] == subject]).reshape(-1)[2:]
        gender = list(df_sex[df_sex['ID'] == subject]['Sex'])
        sex = int(gender[0] == 'FEMALE')

        X1.append(list(features[:182]))
        X2.append(list(features[182:]))
        y_human.append(sex)

X1_human = np.array(X1)
X2_human = np.array(X2)
X1_human = imp_mean.fit_transform(X1_human)
X2_human = imp_mean.fit_transform(X2_human)
print(X1_human.shape, X2_human.shape)

y_human = np.array(y_human)
y_monkey = np.array(y_monkey)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 592/592 [00:00<00:00, 1233.99it/s]


(592, 182) (592, 200)


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14465/14465 [00:31<00:00, 465.37it/s]


(10648, 182) (10648, 200)


## ClassificationProgressiveLearner

In [ ]:
###Progressive Markov
reps = 1
accuracy1 = 0.0
accuracy2 = 0.0

default_transformer_class = TreeClassificationTransformer
default_transformer_kwargs = {
    "kwargs": {"max_depth": 30, "max_features": "sqrt"}
}

default_voter_class = TreeClassificationVoter
default_voter_kwargs = {}

default_decider_class = SimpleArgmaxAverage

for ii in tqdm(range(reps)):
    x_train2, x_test2, y_train2, y_test2 = train_test_split(
                        X1_monkey, y_monkey, train_size=0.8, random_state=0, stratify=y_monkey)
        
    progressive_learner = ProgressiveLearner(
        default_transformer_class=default_transformer_class,
        default_transformer_kwargs=default_transformer_kwargs,
        default_voter_class=default_voter_class,
        default_voter_kwargs=default_voter_kwargs,
        default_decider_class=default_decider_class,
    )
    progressive_learner.add_task(
                X=X1_human,
                y=y_human,
                task_id=0,
                num_transformers=1000,
                transformer_voter_decider_split=[0.67, 0.33, 0],
                decider_kwargs={
                    "classes": np.unique(
                        y_human
                    )
                },
            )
        
    progressive_learner.add_task(
                X=x_train2,
                y=y_train2,
                task_id=1,
                num_transformers=1000,
                transformer_voter_decider_split=[0.67, 0.33, 0],
                decider_kwargs={
                    "classes": np.unique(
                        y_train2
                    )
                },
            )    
    #test on maca (Markov)
    multitask_label = progressive_learner.predict(x_test2, task_id=1, transformer_ids=[0,1])
    singletask_label = progressive_learner.predict(x_test2, task_id=1, transformer_ids=[1])
    accuracy1 += np.mean(multitask_label==y_test2)
    accuracy2 += np.mean(singletask_label==y_test2)
    # Print each iteration's result
    print(f"Iteration {ii+1}, Markov Multitask accuracy:", np.mean(multitask_label == y_test2))
    print(f"Iteration {ii+1}, Markov Singletask accuracy:", np.mean(singletask_label == y_test2))
        
print('Average Markov Multitask accuracy:', accuracy1 / reps)
print('Average Markov Singletask accuracy:', accuracy2 / reps)

In [ ]:
###Progressive Schaefer
reps = 1
accuracy1 = 0.0
accuracy2 = 0.0

default_transformer_class = TreeClassificationTransformer
default_transformer_kwargs = {
    "kwargs": {"max_depth": 30, "max_features": "sqrt"}
}

default_voter_class = TreeClassificationVoter
default_voter_kwargs = {}

default_decider_class = SimpleArgmaxAverage

for ii in tqdm(range(reps)):
    x_train2, x_test2, y_train2, y_test2 = train_test_split(
                        X2_monkey, y_monkey, train_size=0.8, random_state=0, stratify=y_monkey)
    
    progressive_learner = ProgressiveLearner(
    default_transformer_class=default_transformer_class,
    default_transformer_kwargs=default_transformer_kwargs,
    default_voter_class=default_voter_class,
    default_voter_kwargs=default_voter_kwargs,
    default_decider_class=default_decider_class,
)
    progressive_learner.add_task(
                X=X2_human,
                y=y_human,
                task_id=0,
                num_transformers=1000,
                transformer_voter_decider_split=[0.67, 0.33, 0],
                decider_kwargs={
                    "classes": np.unique(
                        y_human
                    )
                },
            )
    
    progressive_learner.add_task(
                X=x_train2,
                y=y_train2,
                task_id=1,
                num_transformers=1000,
                transformer_voter_decider_split=[0.67, 0.33, 0],
                decider_kwargs={
                    "classes": np.unique(
                        y_train2
                    )
                },
            )
    #test on maca (Schaefer)
    multitask_label = progressive_learner.predict(x_test2, task_id=1)
    singletask_label = progressive_learner.predict(x_test2, task_id=1, transformer_ids=[1])
    accuracy1 += np.mean(multitask_label==y_test2)
    accuracy2 += np.mean(singletask_label==y_test2)
    # Print each iteration's result
    print(f"Iteration {ii+1}, Schaefer Multitask accuracy:", np.mean(multitask_label == y_test2))
    print(f"Iteration {ii+1}, Schaefer Singletask accuracy:", np.mean(singletask_label == y_test2))
    
print('Average Schaefer Multitask accuracy:', accuracy1 / reps)
print('Average Schaefer Singletask accuracy:', accuracy2 / reps)

In [ ]:
###Progressive Markov+Schaefer
reps = 1
accuracy1 = 0.0
accuracy2 = 0.0

default_transformer_class = TreeClassificationTransformer
default_transformer_kwargs = {
    "kwargs": {"max_depth": 30, "max_features": "sqrt"}
}

default_voter_class = TreeClassificationVoter
default_voter_kwargs = {}

default_decider_class = SimpleArgmaxAverage

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    np.hstack((X1_monkey, X2_monkey)), y_monkey, train_size=0.8, random_state=0, stratify=y_monkey)
    progressive_learner = ProgressiveLearner(
    default_transformer_class=default_transformer_class,
    default_transformer_kwargs=default_transformer_kwargs,
    default_voter_class=default_voter_class,
    default_voter_kwargs=default_voter_kwargs,
    default_decider_class=default_decider_class,
)
    progressive_learner.add_task(
                X=np.hstack((X1_human,X2_human)),
                y=y_human,
                task_id=0,
                num_transformers=1000,
                transformer_voter_decider_split=[0.67, 0.33, 0],
                decider_kwargs={
                    "classes": np.unique(
                        y_human
                    )
                },
            )
    
    progressive_learner.add_task(
                X=x_train,
                y=y_train,
                task_id=1,
                num_transformers=1000,
                transformer_voter_decider_split=[0.67, 0.33, 0],
                decider_kwargs={
                    "classes": np.unique(
                        y_train
                    )
                },
            )
    #test on maca (Markov+Schaefer)
    multitask_label = progressive_learner.predict(x_test, task_id=1)#train on hum
    singletask_label = progressive_learner.predict(x_test, task_id=1, transformer_ids=[1])#train on maca
    accuracy1 += np.mean(multitask_label==y_test)
    accuracy2 += np.mean(singletask_label==y_test)
    # Print each iteration's result
    print(f"Iteration {ii+1}, Schaefer Multitask accuracy:", np.mean(multitask_label == y_test))
    print(f"Iteration {ii+1}, Schaefer Singletask accuracy:", np.mean(singletask_label == y_test))
    
print('Average Markov+Schaefer Multitask accuracy:', accuracy1 / reps)
print('Average Markov+Schaefer Singletask accuracy:', accuracy2 / reps)

## LifelongClassificationForest

In [ ]:
###Lifelong Markov
reps = 1
accuracy1 = 0.0
accuracy2 = 0.0

for ii in tqdm(range(reps)):
    x_train2, x_test2, y_train2, y_test2 = train_test_split(
                        X1_monkey, y_monkey, train_size=0.8, random_state=0, stratify=y_monkey)
    
    Lifelong_Forest = LifelongClassificationForest()
    Lifelong_Forest.add_task(
                X=X1_human,
                y=y_human,
                task_id=0,
                n_estimators='default',
                tree_construction_proportion='default',
                kappa='default',
                max_depth='default'
            )
    
    Lifelong_Forest.add_task(
                X=x_train2,
                y=y_train2,
                task_id=1,
                n_estimators='default',
                tree_construction_proportion='default',
                kappa='default',
                max_depth='default'
            )
    #test on maca (Markov)
    multitask_label = Lifelong_Forest.predict(x_test2, task_id=0)
    singletask_label = Lifelong_Forest.predict(x_test2, task_id=1)
    accuracy1 += np.mean(multitask_label==y_test2)
    accuracy2 += np.mean(singletask_label==y_test2)
    # Print each iteration's result
    print(f"Iteration {ii+1}, Markov train on Multitask:", np.mean(multitask_label == y_test2))
    print(f"Iteration {ii+1}, Markov train on Singletask:", np.mean(singletask_label == y_test2))
    
print('Average Markov train on Multitask:', accuracy1 / reps)
print('Average Markov train on Singletask:', accuracy2 / reps)

In [ ]:
###Lifelong Schaefer
reps = 1
accuracy1 = 0.0
accuracy2 = 0.0

for ii in tqdm(range(reps)):
    Lifelong_Forest = LifelongClassificationForest()
    
    x_train2, x_test2, y_train2, y_test2 = train_test_split(
                        X2_monkey, y_monkey, train_size=0.8, random_state=0, stratify=y_monkey)
    
    
    Lifelong_Forest.add_task(
                X=X2_human,
                y=y_human,
                task_id=0,
                n_estimators='default',
                tree_construction_proportion='default',
                kappa='default',
                max_depth='default'
            )
    
    Lifelong_Forest.add_task(
                X=x_train2,
                y=y_train2,
                task_id=1,
                n_estimators='default',
                tree_construction_proportion='default',
                kappa='default',
                max_depth='default'
            )
    #Schaefer
    multitask_label = Lifelong_Forest.predict(x_test2, task_id=0)
    singletask_label = Lifelong_Forest.predict(x_test2, task_id=1)
    accuracy1 += np.mean(multitask_label==y_test2)
    accuracy2 += np.mean(singletask_label==y_test2)
    # Print each iteration's result
    print(f"Iteration {ii+1}, Schaefer train on Multitask:", np.mean(multitask_label == y_test2))
    print(f"Iteration {ii+1}, Schaefer train on Singletask:", np.mean(singletask_label == y_test2))
    
print('Average Schaefer train on Multitask:', accuracy1 / reps)
print('Average Schaefer train on Singletask:', accuracy2 / reps)

In [ ]:
#Lifelong both
reps = 1
accuracy1 = 0.0
accuracy2 = 0.0

for ii in tqdm(range(reps)):
    Lifelong_Forest = LifelongClassificationForest()
    
    x_train, x_test, y_train, y_test = train_test_split(
                    np.hstack((X1_monkey, X2_monkey)), y_monkey, train_size=0.8, random_state=0, stratify=y_monkey)

    Lifelong_Forest.add_task(
                X=np.hstack((X1_human,X2_human)),
                y=y_human,
                task_id=0,
                n_estimators='default',
                tree_construction_proportion='default',
                kappa='default',
                max_depth='default'
            )

    Lifelong_Forest.add_task(
                X=x_train,
                y=y_train,
                task_id=1,
                n_estimators='default',
                tree_construction_proportion='default',
                kappa='default',
                max_depth='default'
            )


    multitask_label = Lifelong_Forest.predict(x_test, task_id=0)
    singletask_label = Lifelong_Forest.predict(x_test, task_id=1)
    accuracy1 += np.mean(multitask_label==y_test)
    accuracy2 += np.mean(singletask_label==y_test)


print('Average Markov+Schaefer Multitask accuracy', accuracy1/reps) 
print('Average Markov+Schaefer Singletask accuracy', accuracy2/reps) 